# 05-medallion

Notebook de exploracion. La logica final va en `src/`.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().resolve().parents[1]))

from common.spark_session import create_spark_session

spark = create_spark_session(app_name="05-medallion")
spark

## Dataset

[Chicago Crimes](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2) (2001-2026,
~8,6M filas, 2,4GB en CSV). Encaja con el objetivo del proyecto: practicar una arquitectura por capas
(Bronze -> Silver -> Gold) sobre un dataset lo bastante grande como para que particionado y escritura
idempotente tengan sentido de verdad, algo que no se nota con los datasets pequeños de los proyectos
anteriores.

# 1. Data Loading

Cargo el CSV crudo de `datasets/04-chicago-crimes/`. Al ser ~2,4GB / 8,6M filas, merece la pena
fijarse en el tiempo de inferencia de esquema (`inferSchema=True` obliga a un primer pase completo
sobre el fichero) y decidir si compensa frente a declarar el schema a mano o usar `samplingRatio`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "common").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("No se encontró la carpeta 'common'.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASETS_PATH = PROJECT_ROOT / "datasets" / "04-chicago-crimes"

for file in DATASETS_PATH.iterdir():
    print(file.name)

In [ ]:
# TODO: cargar el CSV (ojo con inferSchema sobre 2,4GB, y con que lat/long usan coma decimal
# en vez de punto -> revisar cómo se leen esas columnas antes de decidir el tipo)


# 2. Data Inventory

Número de filas y columnas antes de entrar en el schema.

In [ ]:
# TODO: nº de filas y columnas


# 3. Data Structure

Reviso el schema: qué columnas son numéricas, cuáles fechas, cuáles categóricas, y confirmo el
problema del separador decimal en `Latitude`/`Longitude` visto en el head del CSV.

In [ ]:
# TODO: printSchema() + show()


# 4. Data Understanding

Cardinalidad de las categóricas (`Primary Type`, `Description`, `District`, `Arrest`, `Domestic`...)
y distribución temporal (`Date`, `Year`), que es la dimensión central de las preguntas de negocio
en Gold (`crimes_by_month`, `crimes_by_district`, `crimes_by_type`, `arrest_rate`).

In [ ]:
# TODO: cardinality_profile / numeric_summary sobre las columnas relevantes


# 5. Data Quality Assessment

Duplicados, nulos y coordenadas fuera de los límites geográficos de Chicago — son exactamente los
controles que luego van en `quality.py` (`assert_no_nulls`, `assert_valid_coordinates`) y en los
pasos de Silver (`deduplicate`, `validate_coordinates`, `handle_nulls`), así que lo que confirme
aquí es lo que despues codifico ahí.

In [ ]:
# TODO: duplicate_count / missing_values_profile


# 6. Bronze -> Silver -> Gold

Aquí es donde practico la arquitectura Medallion en sí: aplano/normalizo en Bronze -> Silver
(tipado, deduplicación, columnas de metadatos de ingestión) y agrego en Gold. La lógica que
valide aquí es la que muevo a `src/bronze.py`, `src/silver.py` y `src/gold.py` en el sprint de
refactor; de momento exploro directamente sobre el DataFrame.

In [ ]:
# TODO: Bronze - metadatos de ingestión (ingestion_timestamp, source_file, ingestion_date)


In [ ]:
# TODO: Silver - normalize_columns (snake_case), cast_types (fechas, lat/long con coma decimal),
# deduplicate, validate_coordinates, handle_nulls


# 7. Business Questions (Gold)

- ¿Cómo evoluciona el nº de delitos por mes/año?
- ¿Qué distritos concentran más incidentes?
- ¿Cuáles son los tipos de delito (`Primary Type`) más frecuentes?
- ¿Cuál es la tasa de arrestos (`Arrest`) global y por tipo de delito?

In [ ]:
# TODO: crimes_by_month


In [ ]:
# TODO: crimes_by_district


In [ ]:
# TODO: crimes_by_type


In [ ]:
# TODO: arrest_rate
